# ДЗ 7. Минимальный мультиагентный прототип на LangGraph

Прототип показывает, как **Trip Manager** делегирует структурированную задачу
агенту **Travel Search Agent**, получает найденные предложения и формирует
итоговый ответ.

Подробнее:

- Manager и Searcher имеют разные ответственности.
- Manager передаёт структурированный `SearchTask`, а не всю историю диалога.
- Только Searcher вызывает поисковый инструмент, и только на чтение.
- Результат возвращается Manager через типизированное состояние LangGraph.
- Условное ребро и `recursion_limit` гарантируют завершение графа.
- Пустой результат является штатным исходом и превращается в понятный ответ.

Локально команда `uv sync` заранее устанавливает зафиксированную версию, поэтому ячейка только печатает сведения об окружении.

В чистой Google Colab VM эта же ячейка устанавливает LangGraph перед выполнением остальных блоков.

In [1]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

EXPECTED_LANGGRAPH_VERSION = "1.2.11"

try:
    installed_langgraph_version = version("langgraph")
except PackageNotFoundError:
    installed_langgraph_version = None

if installed_langgraph_version != EXPECTED_LANGGRAPH_VERSION:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"langgraph=={EXPECTED_LANGGRAPH_VERSION}",
        ]
    )
    installed_langgraph_version = version("langgraph")

print(f"LangGraph {installed_langgraph_version}; Python {sys.version.split()[0]}")

LangGraph 1.2.11; Python 3.14.4


In [2]:
import operator
from typing import Annotated, Literal, TypedDict


class TripRequest(TypedDict):
    origin: str
    destination: str
    departure_date: str
    max_price: int
    currency: str


class SearchTask(TypedDict):
    task_type: Literal["search_travel_offers"]
    origin: str
    destination: str
    departure_date: str
    max_price: int
    currency: str


class TravelOffer(TypedDict):
    offer_id: str
    provider: str
    origin: str
    destination: str
    departure_date: str
    price: int
    currency: str


class AgentMessage(TypedDict):
    sender: Literal["User", "Manager", "Searcher"]
    recipient: Literal["User", "Manager", "Searcher"]
    kind: str
    content: str


class TripState(TypedDict):
    trip_request: TripRequest
    search_task: SearchTask | None
    offers: list[TravelOffer]
    search_completed: bool
    final_answer: str | None
    trace: Annotated[list[AgentMessage], operator.add]
    tool_calls: Annotated[list[str], operator.add]

## 1. Локальный поисковый инструмент только для чтения

Фиксированный локальный каталог, который поисковый инструмент не изменяет,
заменяет API travel-провайдера. Поиск фильтрует его
по маршруту, дате, валюте и бюджету, поэтому результат не зависит от сети и
остаётся одинаковым при каждом запуске.

In [3]:
TRAVEL_OFFERS: tuple[TravelOffer, ...] = (
    {
        "offer_id": "DP-200",
        "provider": "Победа",
        "origin": "Москва",
        "destination": "Санкт-Петербург",
        "departure_date": "2026-10-15",
        "price": 7200,
        "currency": "RUB",
    },
    {
        "offer_id": "SU-100",
        "provider": "Аэрофлот",
        "origin": "Москва",
        "destination": "Санкт-Петербург",
        "departure_date": "2026-10-15",
        "price": 8500,
        "currency": "RUB",
    },
    {
        "offer_id": "S7-300",
        "provider": "S7 Airlines",
        "origin": "Москва",
        "destination": "Санкт-Петербург",
        "departure_date": "2026-10-15",
        "price": 12500,
        "currency": "RUB",
    },
    {
        "offer_id": "SU-101",
        "provider": "Аэрофлот",
        "origin": "Москва",
        "destination": "Санкт-Петербург",
        "departure_date": "2026-10-16",
        "price": 8100,
        "currency": "RUB",
    },
    {
        "offer_id": "U6-400",
        "provider": "Уральские авиалинии",
        "origin": "Санкт-Петербург",
        "destination": "Москва",
        "departure_date": "2026-10-15",
        "price": 6900,
        "currency": "RUB",
    },
)

In [4]:
def search_travel_offers(task: SearchTask) -> list[TravelOffer]:
    """Вернуть подходящие предложения, не изменяя локальный каталог."""
    matching_offers = [
        dict(offer)
        for offer in TRAVEL_OFFERS
        if offer["origin"] == task["origin"]
        and offer["destination"] == task["destination"]
        and offer["departure_date"] == task["departure_date"]
        and offer["currency"] == task["currency"]
        and offer["price"] <= task["max_price"]
    ]
    return sorted(matching_offers, key=lambda offer: (offer["price"], offer["offer_id"]))

In [5]:
tool_test_task: SearchTask = {
    "task_type": "search_travel_offers",
    "origin": "Москва",
    "destination": "Санкт-Петербург",
    "departure_date": "2026-10-15",
    "max_price": 10_000,
    "currency": "RUB",
}

tool_test_results = search_travel_offers(tool_test_task)
assert [offer["offer_id"] for offer in tool_test_results] == ["DP-200", "SU-100"]
assert all(offer["price"] <= tool_test_task["max_price"] for offer in tool_test_results)
assert len(TRAVEL_OFFERS) == 5
assert tool_test_results[0] is not TRAVEL_OFFERS[0]
print("✅ Локальный поисковый инструмент фильтрует и сортирует предложения.")

✅ Локальный поисковый инструмент фильтрует и сортирует предложения.


## 2. Manager и Searcher

In [6]:
from langgraph.graph import END, START, StateGraph


def manager(state: TripState) -> dict:
    """Вызывает поиск и преобразует полученный результат Searcher в ответ пользователю."""
    request = state["trip_request"]

    if not state["search_completed"]:
        search_task: SearchTask = {
            "task_type": "search_travel_offers",
            "origin": request["origin"],
            "destination": request["destination"],
            "departure_date": request["departure_date"],
            "max_price": request["max_price"],
            "currency": request["currency"],
        }
        return {
            "search_task": search_task,
            "trace": [
                {
                    "sender": "Manager",
                    "recipient": "Searcher",
                    "kind": "search_task",
                    "content": (
                        f"Найти варианты {request['origin']} → {request['destination']} "
                        f"на {request['departure_date']} не дороже "
                        f"{request['max_price']} {request['currency']}."
                    ),
                }
            ],
        }

    offers = state["offers"]
    if offers:
        offer_lines = [
            f"{index}. {offer['provider']} ({offer['offer_id']}): "
            f"{offer['price']} {offer['currency']}"
            for index, offer in enumerate(offers, start=1)
        ]
        final_answer = "Найдены подходящие варианты:\n" + "\n".join(offer_lines)
    else:
        final_answer = (
            f"Подходящих вариантов {request['origin']} → {request['destination']} "
            f"на {request['departure_date']} в бюджете до "
            f"{request['max_price']} {request['currency']} не найдено."
        )

    return {
        "final_answer": final_answer,
        "trace": [
            {
                "sender": "Manager",
                "recipient": "User",
                "kind": "final_answer",
                "content": final_answer,
            }
        ],
    }


def searcher(state: TripState) -> dict:
    """Выполняет поиск и возвращает структурированные предложения."""
    search_task = state["search_task"]
    if search_task is None:
        raise ValueError("Searcher требует SearchTask от Manager")

    offers = search_travel_offers(search_task)
    return {
        "offers": offers,
        "search_completed": True,
        "tool_calls": ["Searcher.search_travel_offers"],
        "trace": [
            {
                "sender": "Searcher",
                "recipient": "Manager",
                "kind": "search_result",
                "content": f"Поиск завершён: найдено предложений — {len(offers)}.",
            }
        ],
    }


def route_after_manager(state: TripState) -> str:
    """Запускает Searcher один раз и завершает после ответа Manager."""
    return END if state["final_answer"] is not None else "searcher"

In [7]:
workflow = StateGraph(TripState)
workflow.add_node("manager", manager)
workflow.add_node("searcher", searcher)
workflow.add_edge(START, "manager")
workflow.add_conditional_edges(
    "manager",
    route_after_manager,
    {"searcher": "searcher", END: END},
)
workflow.add_edge("searcher", "manager")
trip_graph = workflow.compile()


def run_trip(request: TripRequest) -> TripState:
    """Выполнить один изолированный поиск поездки с конечным лимитом шагов графа."""
    initial_state: TripState = {
        "trip_request": request,
        "search_task": None,
        "offers": [],
        "search_completed": False,
        "final_answer": None,
        "trace": [
            {
                "sender": "User",
                "recipient": "Manager",
                "kind": "trip_request",
                "content": (
                    f"Нужна поездка {request['origin']} → {request['destination']} "
                    f"на {request['departure_date']}, бюджет — "
                    f"{request['max_price']} {request['currency']}."
                ),
            }
        ],
        "tool_calls": [],
    }
    return trip_graph.invoke(initial_state, config={"recursion_limit": 6})

In [8]:
def print_trip_result(title: str, result: TripState) -> None:
    """Напечатать трассу обмена между агентами и итоговый ответ пользователю."""
    print(f"\n{title}")
    print("=" * len(title))
    for message in result["trace"]:
        print(
            f"{message['sender']} → {message['recipient']} "
            f"[{message['kind']}]: {message['content']}"
        )
    print(f"\nВызовы инструментов: {result['tool_calls']}")
    print(f"\nИтог:\n{result['final_answer']}")

## 3. Демонстрационные сценарии

### Сценарий 1: Предложения найдены

In [9]:
happy_request: TripRequest = {
    "origin": "Москва",
    "destination": "Санкт-Петербург",
    "departure_date": "2026-10-15",
    "max_price": 10_000,
    "currency": "RUB",
}

happy_result = run_trip(happy_request)
print_trip_result("Сценарий 1: предложения найдены", happy_result)

expected_trace = [
    ("User", "Manager"),
    ("Manager", "Searcher"),
    ("Searcher", "Manager"),
    ("Manager", "User"),
]

assert [offer["offer_id"] for offer in happy_result["offers"]] == ["DP-200", "SU-100"]
assert happy_result["tool_calls"] == ["Searcher.search_travel_offers"]
assert [(item["sender"], item["recipient"]) for item in happy_result["trace"]] == expected_trace
assert happy_result["search_task"] == {
    "task_type": "search_travel_offers",
    **happy_request,
}
assert all(
    offer["origin"] == happy_request["origin"]
    and offer["destination"] == happy_request["destination"]
    and offer["departure_date"] == happy_request["departure_date"]
    and offer["currency"] == happy_request["currency"]
    and offer["price"] <= happy_request["max_price"]
    for offer in happy_result["offers"]
)
assert happy_result["offers"] == sorted(
    happy_result["offers"],
    key=lambda offer: (offer["price"], offer["offer_id"]),
)
assert happy_result["search_completed"] is True
assert happy_result["final_answer"].startswith("Найдены подходящие варианты:")

print("\n✅ Сценарий 1: Manager и Searcher корректно обменялись сообщениями.")


Сценарий 1: предложения найдены
User → Manager [trip_request]: Нужна поездка Москва → Санкт-Петербург на 2026-10-15, бюджет — 10000 RUB.
Manager → Searcher [search_task]: Найти варианты Москва → Санкт-Петербург на 2026-10-15 не дороже 10000 RUB.
Searcher → Manager [search_result]: Поиск завершён: найдено предложений — 2.
Manager → User [final_answer]: Найдены подходящие варианты:
1. Победа (DP-200): 7200 RUB
2. Аэрофлот (SU-100): 8500 RUB

Вызовы инструментов: ['Searcher.search_travel_offers']

Итог:
Найдены подходящие варианты:
1. Победа (DP-200): 7200 RUB
2. Аэрофлот (SU-100): 8500 RUB

✅ Сценарий 1: Manager и Searcher корректно обменялись сообщениями.


### Сценарий 2: Предложений в бюджете нет

In [10]:
no_results_request: TripRequest = {
    "origin": "Москва",
    "destination": "Санкт-Петербург",
    "departure_date": "2026-10-15",
    "max_price": 5_000,
    "currency": "RUB",
}

no_results_result = run_trip(no_results_request)
print_trip_result("Сценарий 2: предложений в бюджете нет", no_results_result)

expected_trace = [
    ("User", "Manager"),
    ("Manager", "Searcher"),
    ("Searcher", "Manager"),
    ("Manager", "User"),
]

assert no_results_result["offers"] == []
assert no_results_result["tool_calls"] == ["Searcher.search_travel_offers"]
assert [(item["sender"], item["recipient"]) for item in no_results_result["trace"]] == expected_trace
assert no_results_result["search_completed"] is True
assert no_results_result["final_answer"] is not None
assert "не найдено" in no_results_result["final_answer"].lower()

print("\n✅ Сценарий 2: пустой результат корректно превращён в понятный ответ.")


Сценарий 2: предложений в бюджете нет
User → Manager [trip_request]: Нужна поездка Москва → Санкт-Петербург на 2026-10-15, бюджет — 5000 RUB.
Manager → Searcher [search_task]: Найти варианты Москва → Санкт-Петербург на 2026-10-15 не дороже 5000 RUB.
Searcher → Manager [search_result]: Поиск завершён: найдено предложений — 0.
Manager → User [final_answer]: Подходящих вариантов Москва → Санкт-Петербург на 2026-10-15 в бюджете до 5000 RUB не найдено.

Вызовы инструментов: ['Searcher.search_travel_offers']

Итог:
Подходящих вариантов Москва → Санкт-Петербург на 2026-10-15 в бюджете до 5000 RUB не найдено.

✅ Сценарий 2: пустой результат корректно превращён в понятный ответ.
